In [0]:
%fs
ls 

In [0]:
%sh
ls -l /dbfs/FileStore/tables

In [0]:
%sh
head /dbfs/FileStore/tables/*.csv

In [0]:
%pip install pyarrow pandas deltalake

# Write to delta without pyspark
To do: use timestamp instead of string

In [0]:
import os
import sys
import time
import pandas as pd
#from datetime import datetime
# import duckdb
import pyarrow.compute as pc
from pyarrow import Table
from deltalake import DeltaTable
from deltalake.writer import write_deltalake
from datetime import datetime

In [0]:
dbutils.fs.ls('dbfs:/mnt/bd/openuniverselake/aisraw/type2poc/')

In [0]:
%sh
ls -l /dbfs/mnt/bd/openuniverselake/aisraw/type2poc

In [0]:
csv_path = '/dbfs/mnt/bd/openuniverselake/aisraw/type2poc/csv'
delta_path = '/mnt/bd/openuniverselake/aisraw/type2poc/deltadir'

In [0]:
def read_csvdata(file_name):
    print(f'Read from {file_name} :')
    df = pd.read_csv(file_name, sep=';')
    time.sleep(1)
    #print(df)
    return df


def read_delta():
    dt = DeltaTable(delta_path)
    #print('Delta table:')
    #print(dt.to_pandas())
    return dt


def merge_data(file_name):
    print('Merge new data...')
    new_df = read_csvdata(file_name)
  
    for participant in new_df['participant'].unique():
        print(f"Merge {participant = }")
        # is predicate push down happening here or is all data retrieved and then filtered?
        old_data = read_delta().to_pyarrow_dataset().to_table(filter=( (pc.field("participant") == participant)) ).to_pandas()  

        partition_data = pd.concat([new_df, old_data])
        partition_data = partition_data.drop_duplicates()
        partition_data = Table.from_pandas(partition_data, preserve_index=False) # table frame

        write_deltalake(table_or_uri = delta_path, data = partition_data, mode = "overwrite" , predicate = f"participant='{participant}'" )


def show_result():
    # to do: check for unsorted input data: will the below behave correctly?
    sort_cols = ['participant', 'rec_date', 'inserted_date']
    print('Result:')
    df = read_delta().to_pandas()
    df['valid_from'] = df['inserted_date']
    df['valid_to'] = df.groupby(['participant', 'rec_date'])['inserted_date'].shift(1)  # lead(inserted_date) OVER (PARTITION BY participant, rec_date ORDER BY inserted_date)
    df['valid_to'] = df['valid_to'].fillna('9999-12-31T00:00:00Z')         # coalesce
    display(df.sort_values(by = sort_cols))



def main(run_mode, file_name):
    print(f'Work mode {run_mode} for {file_name}')
    if run_mode == 'init':
        init_data = read_csvdata(file_name)  # pandas data frame
        init_data = Table.from_pandas(init_data, preserve_index=False) # table frame, avoid: Schema contains duplicate qualified field name source.__index_level_0__
        write_deltalake(delta_path, init_data, mode="overwrite", partition_by=['participant']) # create new Delta files
    else:
        merge_data(file_name)
    show_result()

## Adding data

In [0]:
main('init', os.path.join(csv_path, 'initdata.csv'))

In [0]:
for i in range(0, 2):  # do it twice
   main('add', '/dbfs/FileStore/tables/newdata1.csv')

In [0]:
main('add', '/dbfs/FileStore/tables/newdata2.csv') # again

# But I love SQL ❤️

In [0]:
%sql
SELECT * FROM delta.`dbfs:/mnt/bd/openuniverselake/aisraw/type2poc/deltadir`;

In [0]:
%sql
SELECT * FROM delta.`/mnt/bd/openuniverselake/aisraw/type2poc/deltadir`;

In [0]:
%sql
MSCK REPAIR TABLE delta.`dbfs:/mnt/bd/openuniverselake/aisraw/type2poc/deltadir`;

In [0]:
extra_delta_path = '/mnt/bd/openuniverselake/aisraw/type2poc/xdeltadir'
df = read_delta().to_pandas()
spark_df = spark.createDataFrame(df)
spark_df.createOrReplaceTempView("new_data")

In [0]:
%sql
SELECT * FROM new_data;

In [0]:
spark_df.write.format("delta").mode("overwrite").save("/mnt/bd/openuniverselake/aisraw/type2poc/xdeltadir")

In [0]:
%sql
SELECT * FROM delta.`dbfs:/mnt/bd/openuniverselake/aisraw/type2poc/xdeltadir`; /* notice x */

In [0]:
%sql
INSERT OVERWRITE delta.`dbfs:/mnt/bd/openuniverselake/aisraw/type2poc/xdeltadir` 
SELECT * FROM delta.`dbfs:/mnt/bd/openuniverselake/aisraw/type2poc/xdeltadir` 
UNION ALL
SELECT '1973-06-30', 'XXX', 777, NOW()
;

In [0]:
%sql
SELECT * FROM delta.`dbfs:/mnt/bd/openuniverselake/aisraw/type2poc/xdeltadir`; /* notice x */

In [0]:
%sql
DELETE FROM delta.`dbfs:/mnt/bd/openuniverselake/aisraw/type2poc/xdeltadir`
WHERE participant='XXX'
; /* notice x */

In [0]:
%sql
SELECT *
     , lead(inserted_date) over (partition by rec_date, participant order by inserted_date) as valid_to
FROM delta.`dbfs:/mnt/bd/openuniverselake/aisraw/type2poc/xdeltadir`

In [0]:
%sql
SELECT x.rec_date, x.participant, x.val, x.inserted_date
FROM (
      SELECT *
          , lead(inserted_date) over (partition by rec_date, participant order by inserted_date) as valid_to
      FROM delta.`dbfs:/mnt/bd/openuniverselake/aisraw/type2poc/xdeltadir`
) x
WHERE x.valid_to is null and participant='xyz';
    


In [0]:
%sql
INSERT INTO TABLE delta.`dbfs:/mnt/bd/openuniverselake/aisraw/type2poc/xdeltadir`
REPLACE WHERE participant='xyz'
SELECT x.rec_date, x.participant, x.val, x.inserted_date
FROM (
      SELECT *
          , lead(inserted_date) over (partition by rec_date, participant order by inserted_date) as valid_to
      FROM delta.`dbfs:/mnt/bd/openuniverselake/aisraw/type2poc/xdeltadir`
) x
WHERE x.valid_to is null and x.participant='xyz';

In [0]:
%sql
SELECT * FROM delta.`dbfs:/mnt/bd/openuniverselake/aisraw/type2poc/xdeltadir` ORDER BY rec_date, participant; /* notice x */